In [ ]:
import torch
from transformers import PaliGemmaForConditionalGeneration, PaliGemmaProcessor, BitsAndBytesConfig
from PIL import Image
import requests

# 1. Pfadkonfiguration
model_id = "./models/paligemma2-10b-mix-448"

# 2. 8-Bit-Optimierung für die 4090
# Hinweis: Im 8-Bit-Modus werden keine bnb_4bit_... Parameter verwendet
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False,
)

# 3. Prozessor und Modell laden
print("Loading...")

processor = PaliGemmaProcessor.from_pretrained(model_id)

model = PaliGemmaForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16, 
    device_map={"": 0}           
).eval()

print("✅ Modell erfolgreich in den VRAM der 4090 geladen!")

In [ ]:
image = Image.open("Images/86730.png")
image.show()

In [ ]:
# 4. Bild vorbereiten (du kannst auch deinen eigenen lokalen Pfad verwenden, z. B. Image.open("test.jpg"))
image = Image.open("Images/04591.png")

# 5. Prompt festlegen
# Typische Nutzung von PaliGemma: 'caption en' (Beschreibung), 'detect [object]' (Erkennung), 'ocr' (Texterkennung)
## prompt = "What does the combination of the image and text convey?"  # Du kannst diesen Prompt nach Bedarf ändern
prompt = "Is this meme hateful or not? Answer with yes or no." 
## prompt = "caption en"

# 6. Eingabe verarbeiten
inputs = processor(text=prompt, images=image, return_tensors="pt").to(model.device)

# 7. Ausgabe generieren
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=100)

# 8. Ausgabe dekodieren
result = processor.decode(output[0], skip_special_tokens=True)
print(f"\n📝 Output: \n{result[len(prompt):].strip()}")
